In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as f 
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "stgecommerceeastus001", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
df = spark.readStream \
    .format('delta') \
    .option('readChangeFeed', 'true') \
    .table(f"{catalog_name}.silver.slv_order_items")



In [0]:
df_union = df.filter("_change_type IN ('insert', 'update_postimage')")

In [0]:
df_union = df_union.withColumn(
    "gross_amount",
    f.col('quantity') * f.col('unit_price')
)


In [0]:
df_union = df_union.withColumn(
    "discount_amount",
    f.ceil(f.col('gross_amount') * f.col("discount_pct") / 100.0)
)

In [0]:
df_union = df_union.withColumn(
    'sales_amount',
    f.col('gross_amount') - f.col('discount_amount') + f.col("tax_amount")
)

In [0]:
df_union = df_union.withColumn("date_id", f.date_format(f.col("dt"), "yyyyMMdd").cast(IntegerType()))  # Create date_key


df_union = df_union.withColumn(
    "coupon_flag",
    f.when(f.col("coupon_code").isNotNull(), f.lit(1))
     .otherwise(f.lit(0))
)

In [0]:
orders_gold_df = df_union.select(
    f.col("date_id"),
    f.col("dt").alias("transaction_date"),
    f.col("order_ts").alias("transaction_ts"),
    f.col("order_id").alias("transaction_id"),
    f.col("customer_id"),
    f.col("item_seq").alias("seq_no"),
    f.col("product_id"),
    f.col("channel"),
    f.col("coupon_code"),
    f.col("coupon_flag"),
    f.col("unit_price_currency"),
    f.col("quantity"),
    f.col("unit_price"),
    f.col("gross_amount"),
    f.col("discount_pct").alias("discount_percent"),
    f.col("discount_amount"),
    f.col("tax_amount"),
    f.col("sales_amount").alias("net_amount")
)

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_items/"
print(gold_checkpoint_path)

In [0]:
def upsert_to_gold(microBatchDF, batchId):
    table_name = f"{catalog_name}.gold.gld_fact_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("gold_table").merge(
            microBatchDF.alias("batch_table"),
            "gold_table.transaction_id = batch_table.transaction_id AND gold_table.seq_no = batch_table.seq_no",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

orders_gold_df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_gold
).option("checkpointLocation", gold_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).start().awaitTermination()

In [0]:
spark.sql(f"SELECT count(*) FROM {catalog_name}.gold.gld_fact_order_items").show()